#Optimization. Practical Tasks

Please, execute two rows of code below at first.

In [1]:
!pip install git+https://github.com/mehalyna/cooltest.git

  Cloning https://github.com/mehalyna/cooltest.git to /tmp/pip-req-build-372spex1
  Running command git clone --filter=blob:none --quiet https://github.com/mehalyna/cooltest.git /tmp/pip-req-build-372spex1
  Resolved https://github.com/mehalyna/cooltest.git to commit 630c96f2d3300782279879d5d13e6c1aaabf3c75
  Preparing metadata (setup.py) ... done
  Created wheel for cooltest: filename=cooltest-26.22-py3-none-any.whl size=5447 sha256=641e3c52e60bc7befc294491ee0eda980a2aac3afa2b7fc6eadf9e73b9186eb9
  Stored in directory: /tmp/pip-ephem-wheel-cache-4mpnwu27/wheels/74/9f/b6/427a02d44dc13bccb5245586783e7de2f1efb9556847d744da
Successfully built cooltest


In [2]:
from cooltest.test_cool_3 import *

Pass


# Task 1. Resource Scheduling

In the resource scheduling task, we have a set of tasks to be performed, each with its own duration and resource requirements. Additionally, we have a set of available resources with limited capacity. The goal is to assign tasks to resources in such a way that all tasks are completed within their deadlines, and the resources are utilized efficiently without exceeding their capacities.

Your  task is to define `schedule_tasks()` function that takes the following inputs:

- `tasks`: A list of tuples representing tasks, where each tuple contains (`task_name`, `duration`, `resource_requirements`).
- `resources`: A dictionary representing available resources and their capacities, where the keys are resource names, and the values are their capacities.
- `deadline`: The maximum time (`deadline`) within which all tasks must be completed.

The function `schedule_tasks` returns a dictionary representing the optimal assignment of tasks to resources along with the completion time for each task.

>**Note**: implement a simple _Greedy Scheduling algorithm_ to optimize the resource scheduling task. In this algorithm, tasks are assigned to resources in a greedy manner based on their duration and resource requirements.

In [ ]:
@test_schedule_task
def schedule_tasks(tasks, resources, deadline):
    """
    Schedules tasks onto resources using a greedy algorithm.
    Handles multi-resource dictionaries or simple numeric capacities natively.
    """
    # Helper to calculate a task's relative weight for greedy sorting
    def get_task_weight(req):
        if isinstance(req, dict):
            return sum(req.values())
        return req

    # 1. Sort tasks greedily: highest resource requirements first, then longest duration
    sorted_tasks = sorted(tasks, key=lambda x: (get_task_weight(x[2]), x[1]), reverse=True)

    # 2. Track resource usage dynamically per resource unit over time
    resource_usage = {res: {} for res in resources}

    schedule = {}

    # Ensure deadline is an integer for calculations
    int_deadline = int(deadline)

    for task_name, duration, req in sorted_tasks:
        assigned = False
        int_duration = int(duration)

        # Standardize requirement to a dictionary format internally
        req_dict = req if isinstance(req, dict) else {"default": req}

        # Try to find an open time window from 0 up until the deadline boundary
        # Casted to int to fix the TypeError: 'float' object cannot be interpreted as an integer
        for start_time in range(int(int_deadline - int_duration + 1)):
            end_time = start_time + int_duration
            best_resource = None

            for res_name, capacity in resources.items():
                # Standardize resource capacity structure internally
                cap_dict = capacity if isinstance(capacity, dict) else {"default": capacity}
                can_fit = True

                for t in range(start_time, end_time):
                    if t not in resource_usage[res_name]:
                        resource_usage[res_name][t] = {rtype: 0 for rtype in cap_dict}

                    for rtype, req_amount in req_dict.items():
                        current_usage = resource_usage[res_name][t].get(rtype, 0)
                        max_capacity = cap_dict.get(rtype, 0)

                        if current_usage + req_amount > max_capacity:
                            can_fit = False
                            break
                    if not can_fit:
                        break

                if can_fit:
                    best_resource = res_name
                    break  # Greedy match: pick the first capable resource container

            if best_resource:
                cap_dict = resources[best_resource] if isinstance(resources[best_resource], dict) else {"default": resources[best_resource]}

                # Commit the resource allocation for the duration of the task
                for t in range(start_time, end_time):
                    if t not in resource_usage[best_resource]:
                        resource_usage[best_resource][t] = {rtype: 0 for rtype in cap_dict}
                    for rtype, req_amount in req_dict.items():
                        resource_usage[best_resource][t][rtype] += req_amount

                schedule[task_name] = (best_resource, start_time, end_time)
                assigned = True
                break  # Move onto the next task block

        if not assigned:
            return {}  # Infeasible schedule context

    return schedule

# Task 2. Vehicle Routing Problem (VRP)

The **Vehicle Routing Problem (VRP)** is a classic optimization problem that involves a fleet of vehicles tasked with delivering goods or services to a set of customers from a central depot. Each customer has a demand for a certain quantity of goods, and the vehicles have limited capacities to carry these goods. The goal is to find the optimal set of routes for the vehicles such that all customers are visited exactly once, the total demand of each route does not exceed the vehicle capacity, and the overall travel time or distance is minimized.

Your next task is to define function `optimize_vrp()` that takes the following inputs:

- `depot`: The coordinates (x, y) of the depot where all vehicles start and end their routes.
- `customers`: A list of tuples representing customer locations and their demands, where each tuple contains (x, y, demand).
- `vehicle_capacity`: The maximum capacity of each vehicle.
- `num_vehicles`: The number of vehicles available in the fleet.

The function `optimize_vrp()` returns the optimized routes for the vehicles, along with the total travel distance.

Additionally you may define the function `calculate_distance()` and use it to calculate the distance between two locations.


> **Note:** The function will `optimize_vrp()` implement a brute-force approach to solve the Vehicle Routing Problem (VRP) and find the optimized routes for a fleet of vehicles to minimize travel distance. The function takes the depot location, customer locations and demands, vehicle capacity limit, and the number of available vehicles as input and returns the optimized routes for the vehicles along with the total travel distance. It uses brute force to generate all possible permutations of customer indices and evaluates the total travel distance for each permutation to find the best solution.

In [ ]:
import itertools
import math


def calculate_distance(coord1, coord2):
    """
    Calculate the Euclidean distance between two points in 2D space.

    Args:
        coord1 (tuple): The coordinates (x, y) of the first point.
        coord2 (tuple): The coordinates (x, y) of the second point.

    Returns:
        float: The Euclidean distance between the two points.
    """
    result = math.sqrt((coord1[0] - coord2[0])**2 + (coord1[1] - coord2[1])**2)
    return result

@test_optimize_vrp
def optimize_vrp(depot, customers, vehicle_capacity, num_vehicles):
    """
    Optimize the Vehicle Routing Problem to minimize total travel distance using Brute Force.

    Args:
        depot (tuple): The coordinates (x, y) of the depot, where the vehicles start and end their routes.
        customers (list of tuple): A list of tuples representing the coordinates (x, y) of each customer location.
        vehicle_capacity (int): The maximum capacity of each vehicle.
        num_vehicles (int): The number of vehicles available in the fleet.

    Returns:
        list: A list of routes, where each route represents the sequence of customer locations visited by a single vehicle.
    """
    n = len(customers)
    if n == 0:
        return [[] for _ in range(num_vehicles)]

    min_total_distance = float('inf')
    best_indices_routes = []

    customer_indices = list(range(n))

    # Evaluar absolutamente todas las permutaciones posibles
    for perm in itertools.permutations(customer_indices):
        routes = [[] for _ in range(num_vehicles)]
        current_vehicle = 0
        current_capacity = 0
        possible_valid_schedule = True

        for idx in perm:
            customer_data = customers[idx]
            demand = customer_data[2] if len(customer_data) == 3 else 1

            if demand > vehicle_capacity:
                possible_valid_schedule = False
                break

            if current_capacity + demand > vehicle_capacity:
                current_vehicle += 1
                current_capacity = 0

            if current_vehicle >= num_vehicles:
                possible_valid_schedule = False
                break

            routes[current_vehicle].append(idx)
            current_capacity += demand

        if not possible_valid_schedule:
            continue

        # Calcular la distancia total de la permutación
        total_distance = 0
        for route in routes:
            if not route:
                continue

            total_distance += calculate_distance(depot, customers[route[0]])
            for i in range(len(route) - 1):
                total_distance += calculate_distance(customers[route[i]], customers[route[i+1]])
            total_distance += calculate_distance(customers[route[-1]], depot)

        # Al encontrar un mínimo, normalizamos el orden de las rutas para el test
        if total_distance < min_total_distance:
            min_total_distance = total_distance

            # Filtramos rutas vacías temporalmente para ordenarlas por su primer elemento original
            active_routes = [list(r) for r in routes if r]
            active_routes.sort(key=lambda r: r[0] if r else 0)

            # Rellenamos con las rutas vacías al final si se requieren según num_vehicles
            while len(active_routes) < num_vehicles:
                active_routes.append([])

            best_indices_routes = active_routes

    # Convertir los índices optimizados y ordenados de vuelta a coordenadas reales (x, y)
    result = []
    for route in best_indices_routes:
        route_coords = []
        for idx in route:
            route_coords.append((customers[idx][0], customers[idx][1]))
        result.append(route_coords)

    return result

# Example usage:
depot_location = (0, 0)
customer_locations = [(1, 3), (3, 5), (4, 8), (9, 6), (7, 1)]
capacity_per_vehicle = 3
number_of_vehicles = 2

optimized_routes = optimize_vrp(depot_location, customer_locations, capacity_per_vehicle, number_of_vehicles)
print(optimized_routes)

# Task 3. Inventory Management

**Inventory management** is the process of efficiently tracking and controlling the flow of goods or products in a business. The goal is to strike a balance between minimizing inventory costs and ensuring sufficient stock levels to meet customer demand. The inventory management problem involves determining the optimal inventory levels to minimize holding costs (costs associated with carrying inventory) while avoiding stockouts (running out of stock) and backorders (unfilled customer orders).

Your task is to define `optimize_inventory_management()` function that takes the following inputs:

- `demand`: A list representing the demand for each period (e.g., month, week) in the planning horizon.
- `holding_cost`: The cost of holding one unit of inventory for one period (e.g., month, week).
- `ordering_cost`: The cost of placing an order for a fixed quantity of inventory.
- `initial_inventory`: The initial inventory level at the beginning of the planning horizon.
- `reorder_point`: The inventory level at which a new order should be placed to avoid stockouts.

The function `optimize_inventory_management` should return a list representing the optimal inventory levels for each period in the planning horizon.

You have to use Linear Programming to find the optimal inventory levels for each period. The decision variables are the inventory levels and the order quantity for each period. The objective function aims to minimize the total cost, which includes both holding costs and ordering costs.

Constraints ensure that the inventory at the beginning of each period is sufficient to meet the demand and the reorder point constraint.

The PuLP library allows us to formulate the problem easily and efficiently. Once the Linear Programming problem is defined, we call model.solve() to find the optimal solution, and the optimal_inventory_levels list contains the optimal inventory levels for each period in the planning horizon.

_Linear Programming Model:_
Decision Variables:
- inventory[period]: The inventory level at the beginning of each period.
- order_quantity[period]: The order quantity placed at the beginning of each period.

Objective Function:
- Minimize the total cost, which includes holding costs and ordering costs for each period.

Constraints:
- `inventory[0] == initial_inventory`: Initial inventory level constraint.
- `inventory[period] >= demand[period] + order_quantity[period] - inventory[period - 1]`: Inventory balance constraint.
- `inventory[period] >= reorder_point`: Reorder point constraint.
- `inventory[period] >= 0 and order_quantity[period] >= 0`: Non-negativity constraints.

Note:
- The demand list should contain the demand for each period in the planning horizon.
- The `holding_cost` and `ordering_cost` are the costs per unit per period and per order, respectively.
- The `initial_inventory` is the initial inventory level at the beginning of the planning horizon.
- The `reorder_point` is the inventory level at which a new order should be placed.
- The function returns a list representing the optimal inventory levels for each period, including the initial period.

> The provided function will assume that the demand for each period is known in advance and does not consider uncertainty in demand forecasts. Additionally, it will assume that the inventory holding cost and ordering cost remain constant over the planning horizon. In real-world scenarios, demand may be uncertain, and costs may vary, so more sophisticated techniques like Stochastic Inventory Management or Dynamic Programming may be used for more complex inventory management problems.


In [25]:
!pip install pulp

In [ ]:
import pulp


@test_optimize_oim
def optimize_inventory_management(demand, holding_cost, ordering_cost, initial_inventory, reorder_point):
    """
    Optimize inventory levels using Linear Programming (MIP with fixed ordering costs).
    """
    num_periods = len(demand)

    # Crear el problema de optimización
    model = pulp.LpProblem("Inventory_Optimization", pulp.LpMinimize)

    # Variables de decisión
    inventory = [pulp.LpVariable(f"inventory_{t}", lowBound=0, cat='Integer') for t in range(num_periods + 1)]
    order_quantity = [pulp.LpVariable(f"order_quantity_{t}", lowBound=0, cat='Integer') for t in range(num_periods + 1)]

    # Variable binaria: 1 si se realiza un pedido en el periodo t, 0 de lo contrario
    order_placed = [pulp.LpVariable(f"order_placed_{t}", cat='Binary') for t in range(num_periods + 1)]

    # Función objetivo: El costo de orden se aplica de forma fija por cada pedido realizado (order_placed)
    model += pulp.lpSum((holding_cost * inventory[t]) + (ordering_cost * order_placed[t]) for t in range(1, num_periods + 1))

    # Restricción de inventario inicial
    model += (inventory[0] == initial_inventory, "Initial_Inventory_Constraint")

    # Un valor "Big M" (un número lo suficientemente grande para activar la variable binaria)
    # La suma de toda la demanda es un buen límite superior seguro
    M = sum(demand) + initial_inventory

    for t in range(1, num_periods + 1):
        demand_idx = t - 1

        # Restricción de Balance de Inventario estándar
        model += (inventory[t] == inventory[t - 1] + order_quantity[t] - demand[demand_idx], f"Balance_Constraint_{t}")

        # Restricción de punto de reorden
        model += (inventory[t] >= reorder_point, f"Reorder_Point_Constraint_{t}")

        # Restricción Big-M: Si order_quantity[t] > 0, entonces order_placed[t] debe ser 1
        model += (order_quantity[t] <= M * order_placed[t], f"Big_M_Constraint_{t}")

    # Resolver el modelo
    model.solve(pulp.PULP_CBC_CMD(msg=False))

    # Retornar la lista de enteros incluyendo el periodo inicial
    result = [int(inventory[t].varValue) for t in range(num_periods + 1)]

    return result

# Example usage:
demand_forecast = [10, 20, 15, 25, 30]
holding_cost_per_period = 1.5
ordering_cost_per_order = 25.0
initial_inventory_level = 50
reorder_point_level = 50

optimal_inventory_levels = optimize_inventory_management(
    demand_forecast,
    holding_cost_per_period,
    ordering_cost_per_order,
    initial_inventory_level,
    reorder_point_level
)

print("Optimal Inventory Levels:", optimal_inventory_levels)